<a href="https://colab.research.google.com/github/mr-zero-000/Statistical-Learning-e23034/blob/main/Assignment%206/Assignment_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
"""
=============================================================================
Energy Efficiency & Green Building — Complete Analysis  (Interactive)
=============================================================================
Produces all 6 figures as interactive pop-up windows:

  Fig 1  — ENB2012 EDA (scatter plots, correlation heatmap, distributions)
  Fig 2  — GPR results (predicted vs actual, residuals, uncertainty bands)
  Fig 3  — GPR kernel analysis & single-parameter exploration
  Fig 4  — Green Building EDA (feature vs target scatter plots)
  Fig 5  — Linear Regression results (predicted vs actual, residuals, coefficients)
  Fig 6  — Linear Regression diagnostics (correlation heatmap, CV box-plot)

Each figure opens in its own window.  Close a window to proceed to the next.

DATASETS
--------
  • ENB2012  : 768-sample energy efficiency dataset (Tsanas & Xifara 2012)
               https://www.kaggle.com/datasets/elikplim/eergy-efficiency-dataset
  • Green Building: 2400-sample multi-source building environment dataset
               https://www.kaggle.com/datasets/programmer3/green-building-multi-source-environment-dataset

If you have kagglehub installed and credentials configured, uncomment the
kagglehub download blocks below and comment out the "generate synthetic data"
blocks.  The column names and structure are identical either way.

REQUIREMENTS
------------
    pip install numpy pandas matplotlib seaborn scikit-learn scipy

Run:
    python energy_analysis_interactive.py
=============================================================================
"""

# ─── Imports ──────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm as sp_norm

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, Matern, ConstantKernel as C
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

import warnings
warnings.filterwarnings('ignore')

# ─── Global Style ─────────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family'       : 'DejaVu Sans',
    'font.size'         : 10,
    'axes.titlesize'    : 12,
    'axes.labelsize'    : 10,
    'axes.spines.top'   : False,
    'axes.spines.right' : False,
    'figure.dpi'        : 150,
})
BLUE   = '#2563EB'
GREEN  = '#16A34A'
RED    = '#DC2626'
PURPLE = '#7C3AED'
GRAY   = '#6B7280'


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 0  ─  DATA LOADING / GENERATION
# ══════════════════════════════════════════════════════════════════════════════

def load_enb2012():
    """
    Load the ENB2012 energy efficiency dataset.

    To use the real Kaggle dataset, install kagglehub and replace this
    function body with:

        import kagglehub, os, pandas as pd
        path = kagglehub.dataset_download("elikplim/eergy-efficiency-dataset")
        return pd.read_csv(os.path.join(path, "ENB2012_data.csv"))

    The real CSV has columns: X1 X2 X3 X4 X5 X6 X7 X8 Y1 Y2
    """
    np.random.seed(42)

    # Discrete building-parameter grid (from the published paper)
    RC_vals = [0.62, 0.64, 0.66, 0.68, 0.70, 0.71, 0.74, 0.76,
               0.82, 0.86, 0.90, 0.98]
    SA_map  = {0.62:808.5, 0.64:784.0, 0.66:784.0, 0.68:759.5,
               0.70:735.0, 0.71:710.5, 0.74:686.0, 0.76:661.5,
               0.82:637.0, 0.86:612.5, 0.90:563.5, 0.98:514.5}
    WA_map  = {rc: max(220.5, 367.5 - i*25) for i, rc in enumerate(RC_vals)}
    RA_map  = {rc: (220.5 if rc < 0.82 else 147.0 if rc < 0.90 else 122.5)
               for rc in RC_vals}
    OH_map  = {rc: (7.0 if rc < 0.82 else 3.5) for rc in RC_vals}

    rows = []
    for rc in RC_vals:
        sa, wa, ra, oh = SA_map[rc], WA_map[rc], RA_map[rc], OH_map[rc]
        for orient in [2, 3, 4, 5]:
            for ga in [0, 0.1, 0.25, 0.4]:
                for gad in ([0] if ga == 0 else [1, 2, 3, 4, 5]):
                    hl = (30 - 20*rc + 0.05*sa - 0.02*wa + 5*oh
                          + 3*ga + 0.3*gad + np.random.normal(0, 0.5))
                    cl = (18 - 10*rc + 0.03*sa - 0.01*wa + 3*oh
                          + 4*ga + 0.4*gad + 0.5*orient + np.random.normal(0, 0.5))
                    rows.append([rc, sa, wa, ra, oh, orient, ga, gad,
                                 max(hl, 6.01), max(cl, 10.9)])

    df = pd.DataFrame(rows,
                      columns=['X1','X2','X3','X4','X5','X6','X7','X8','Y1','Y2'])
    return df.sample(frac=1, random_state=42).reset_index(drop=True)


def load_green_building():
    """
    Load the Green Building multi-source environment dataset.

    To use the real Kaggle dataset, install kagglehub and replace this
    function body with:

        import kagglehub, os, pandas as pd
        path = kagglehub.dataset_download(
                   "programmer3/green-building-multi-source-environment-dataset")
        return pd.read_csv(os.path.join(path, "green_building_dataset.csv"))

    The real CSV has 13 columns including 'predicted_energy_demand'.
    """
    np.random.seed(42)
    n = 2400

    temp         = np.random.uniform(15,  35,  n)
    humidity     = np.random.uniform(30,  90,  n)
    co2          = np.random.uniform(400, 1200, n)
    light        = np.random.uniform(100, 1000, n)
    occupancy    = np.random.randint(0,   51,  n)
    hvac         = np.random.uniform(10,  100, n)
    solar_rad    = np.random.uniform(0,   800, n)
    wind_sp      = np.random.uniform(0,   10,  n)
    building_age = np.random.randint(1,   51,  n)
    insulation   = np.random.uniform(0.1, 5.0, n)
    window_ratio = np.random.uniform(0.1, 0.9, n)
    floor_area   = np.random.uniform(50,  500, n)

    pred_energy = np.clip(
        0.8*hvac + 0.3*temp - 2.0*insulation
        + 0.05*occupancy + 0.001*floor_area*temp
        + 0.01*solar_rad*window_ratio
        + 0.02*building_age + np.random.normal(0, 2, n),
        5, 200
    )
    return pd.DataFrame({
        'temperature'          : temp,
        'humidity'             : humidity,
        'co2_level'            : co2,
        'light_level'          : light,
        'occupancy_count'      : occupancy,
        'hvac_energy'          : hvac,
        'solar_radiation'      : solar_rad,
        'wind_speed'           : wind_sp,
        'building_age'         : building_age,
        'insulation_thickness' : insulation,
        'window_to_wall_ratio' : window_ratio,
        'floor_area'           : floor_area,
        'predicted_energy_demand': pred_energy,
    })


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 1  ─  GAUSSIAN PROCESS REGRESSION  (ENB2012)
# ══════════════════════════════════════════════════════════════════════════════

def run_gpr(df):
    """Fit GPR models for Y1 (Heating) and Y2 (Cooling) and return all results."""
    FEAT_NAMES = ['Relative\nCompactness', 'Surface\nArea', 'Wall\nArea',
                  'Roof\nArea', 'Overall\nHeight', 'Orientation',
                  'Glazing\nArea', 'Glazing Area\nDist']

    X  = df[['X1','X2','X3','X4','X5','X6','X7','X8']].values
    Y1 = df['Y1'].values
    Y2 = df['Y2'].values

    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)

    # GPR is O(n³) — use 300 training points
    rng  = np.random.RandomState(0)
    idx  = rng.choice(len(Xs), 300, replace=False)
    mask = np.zeros(len(Xs), bool);  mask[idx] = True

    X_tr, X_te   = Xs[mask], Xs[~mask]
    y1_tr, y1_te = Y1[mask], Y1[~mask]
    y2_tr, y2_te = Y2[mask], Y2[~mask]

    kernel = (C(1.0, (1e-3, 1e3))
              * Matern(length_scale=np.ones(8), nu=2.5)
              + WhiteKernel(noise_level=0.5))

    gpr1 = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=5,
                                    normalize_y=True, random_state=42)
    gpr2 = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=5,
                                    normalize_y=True, random_state=42)
    print("  Fitting GPR for Y1 (Heating Load)…")
    gpr1.fit(X_tr, y1_tr)
    print("  Fitting GPR for Y2 (Cooling Load)…")
    gpr2.fit(X_tr, y2_tr)

    y1_pred, y1_std = gpr1.predict(X_te, return_std=True)
    y2_pred, y2_std = gpr2.predict(X_te, return_std=True)

    # Combined target  Y_avg = (Y1+Y2)/2
    y_avg    = (Y1 + Y2) / 2
    k_avg    = (C(1.0) * Matern(length_scale=np.ones(8), nu=2.5) + WhiteKernel(0.5))
    gpr_avg  = GaussianProcessRegressor(k_avg, n_restarts_optimizer=5,
                                        normalize_y=True, random_state=42)
    print("  Fitting GPR for Y_avg = (Y1+Y2)/2…")
    gpr_avg.fit(X_tr, y_avg[mask])
    yp_avg, ys_avg = gpr_avg.predict(X_te, return_std=True)

    # Single-feature GPR  (X5 → Y1 for 1-D visualisation)
    gpr_x5 = GaussianProcessRegressor(
        C(1.0)*RBF(1.0) + WhiteKernel(0.5),
        n_restarts_optimizer=3, normalize_y=True, random_state=0)
    gpr_x5.fit(Xs[:, 4:5][mask], y1_tr)

    # Print metrics
    r2_y1  = r2_score(y1_te, y1_pred)
    r2_y2  = r2_score(y2_te, y2_pred)
    r2_avg_val = r2_score(y_avg[~mask], yp_avg)
    print(f"\n  GPR Y1 : R²={r2_y1:.4f}  "
          f"RMSE={np.sqrt(mean_squared_error(y1_te,y1_pred)):.3f}  "
          f"MAE={mean_absolute_error(y1_te,y1_pred):.3f}")
    print(f"  GPR Y2 : R²={r2_y2:.4f}  "
          f"RMSE={np.sqrt(mean_squared_error(y2_te,y2_pred)):.3f}  "
          f"MAE={mean_absolute_error(y2_te,y2_pred):.3f}")
    print(f"  GPR avg: R²={r2_avg_val:.4f}  "
          f"RMSE={np.sqrt(mean_squared_error(y_avg[~mask],yp_avg)):.3f}")

    return dict(
        X=X, Y1=Y1, Y2=Y2, Xs=Xs, mask=mask,
        X_tr=X_tr, X_te=X_te,
        y1_tr=y1_tr, y1_te=y1_te, y1_pred=y1_pred, y1_std=y1_std,
        y2_tr=y2_tr, y2_te=y2_te, y2_pred=y2_pred, y2_std=y2_std,
        yp_avg=yp_avg, ys_avg=ys_avg, y_avg_te=y_avg[~mask],
        gpr1=gpr1, gpr2=gpr2, gpr_x5=gpr_x5,
        FEAT_NAMES=FEAT_NAMES,
    )


# ── Figure 1 : ENB2012 EDA ────────────────────────────────────────────────────
def fig1_eda(df, res):
    print("\n[Fig 1] ENB2012 EDA…")
    X, Y1, Y2 = res['X'], res['Y1'], res['Y2']
    FEAT_NAMES = res['FEAT_NAMES']

    fig, axes = plt.subplots(3, 4, figsize=(16, 11))
    fig.suptitle('ENB2012 – Exploratory Data Analysis',
                 fontsize=14, fontweight='bold', y=0.98)

    # Row 0 : feature vs loads (first 8 features use first 8 sub-axes)
    for i, ax in enumerate(axes[0]):
        ax.scatter(X[:, i], Y1, alpha=0.3, s=12, color=BLUE,  label='Y1 Heat')
        ax.scatter(X[:, i], Y2, alpha=0.3, s=12, color=RED,   label='Y2 Cool')
        ax.set_xlabel(FEAT_NAMES[i])
        ax.set_ylabel('Load (kWh/m²)' if i == 0 else '')
        ax.set_title(f'X{i+1} vs Loads')
        if i == 0:
            ax.legend(fontsize=8)

    # Row 1 col 0 : correlation heatmap
    cols = [f'X{i}' for i in range(1, 9)] + ['Y1', 'Y2']
    corr = df[cols].corr()
    sns.heatmap(corr, ax=axes[1][0], cmap='RdBu_r', vmin=-1, vmax=1,
                annot=True, fmt='.2f', annot_kws={'size': 6},
                xticklabels=cols, yticklabels=cols, linewidths=0.3,
                cbar_kws={'shrink': 0.7})
    axes[1][0].set_title('Correlation Matrix')
    axes[1][0].tick_params(labelsize=7, rotation=45)

    # Row 1 col 1 : Y1 histogram
    axes[1][1].hist(Y1, bins=30, color=BLUE, alpha=0.7, edgecolor='white')
    axes[1][1].set_title('Heating Load (Y1) Distribution')
    axes[1][1].set_xlabel('kWh/m²')

    # Row 1 col 2 : Y2 histogram
    axes[1][2].hist(Y2, bins=30, color=RED, alpha=0.7, edgecolor='white')
    axes[1][2].set_title('Cooling Load (Y2) Distribution')
    axes[1][2].set_xlabel('kWh/m²')

    # Row 1 col 3 : Y1 vs Y2 scatter
    axes[1][3].scatter(Y1, Y2, alpha=0.3, s=12, color=PURPLE)
    axes[1][3].set_xlabel('Heating Load Y1')
    axes[1][3].set_ylabel('Cooling Load Y2')
    axes[1][3].set_title(f'Y1 vs Y2  (r={np.corrcoef(Y1, Y2)[0,1]:.3f})')

    # Row 2 col 0 : |Pearson r| bar chart
    bar_x = np.arange(8)
    w = 0.35
    axes[2][0].bar(bar_x - w/2, np.abs(corr.iloc[:8, 8]),
                   width=w, color=BLUE, label='|r| with Y1', alpha=0.8)
    axes[2][0].bar(bar_x + w/2, np.abs(corr.iloc[:8, 9]),
                   width=w, color=RED,  label='|r| with Y2', alpha=0.8)
    axes[2][0].set_xticks(bar_x)
    axes[2][0].set_xticklabels([f'X{i+1}' for i in range(8)], fontsize=8)
    axes[2][0].set_title('|Pearson r| with Targets')
    axes[2][0].set_ylabel('|r|')
    axes[2][0].legend(fontsize=8)

    for j in range(1, 4):
        axes[2][j].axis('off')

    plt.tight_layout()
    plt.show()
    print("  Fig 1 displayed.")


# ── Figure 2 : GPR Results ────────────────────────────────────────────────────
def fig2_gpr_results(res):
    print("[Fig 2] GPR results…")
    y1_te, y1_pred, y1_std = res['y1_te'], res['y1_pred'], res['y1_std']
    y2_te, y2_pred, y2_std = res['y2_te'], res['y2_pred'], res['y2_std']

    r2_y1  = r2_score(y1_te, y1_pred)
    rmse_y1= np.sqrt(mean_squared_error(y1_te, y1_pred))
    r2_y2  = r2_score(y2_te, y2_pred)
    rmse_y2= np.sqrt(mean_squared_error(y2_te, y2_pred))

    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    fig.suptitle('Gaussian Process Regression — ENB2012 Energy Efficiency',
                 fontsize=14, fontweight='bold')

    # Columns 0 : Predicted vs Actual (coloured by σ)
    for row, (yte, ypred, ystd, lab, r2, rmse) in enumerate([
            (y1_te, y1_pred, y1_std, 'Heating Load Y1', r2_y1, rmse_y1),
            (y2_te, y2_pred, y2_std, 'Cooling Load Y2', r2_y2, rmse_y2)]):
        ax  = axes[row][0]
        mn  = min(yte.min(), ypred.min())
        mx  = max(yte.max(), ypred.max())
        sc  = ax.scatter(yte, ypred, c=ystd, cmap='plasma', s=20, alpha=0.7)
        ax.plot([mn, mx], [mn, mx], 'k--', lw=1.5, label='Perfect fit')
        ax.set_xlabel(f'Actual {lab}')
        ax.set_ylabel('Predicted')
        ax.set_title(f'{lab}\nR²={r2:.4f}   RMSE={rmse:.3f}')
        ax.legend(fontsize=8)
        plt.colorbar(sc, ax=ax, label='GPR σ')

    # Columns 1 : Residuals vs Fitted
    for row, (yte, ypred, col, lab) in enumerate([
            (y1_te, y1_pred, BLUE, 'Y1'),
            (y2_te, y2_pred, RED,  'Y2')]):
        ax  = axes[row][1]
        res_= ypred - yte
        ax.scatter(ypred, res_, alpha=0.4, s=15, color=col)
        ax.axhline(0, color='k', ls='--', lw=1.5)
        ax.set_xlabel('Fitted values')
        ax.set_ylabel('Residual')
        ax.set_title(f'Residuals — {lab}')

    # Columns 2 : Uncertainty bands (sorted by σ)
    for row, (yte, ypred, ystd, col, lab) in enumerate([
            (y1_te, y1_pred, y1_std, BLUE, 'Heating Load Y1'),
            (y2_te, y2_pred, y2_std, RED,  'Cooling Load Y2')]):
        ax   = axes[row][2]
        sidx = np.argsort(ystd)
        ax.fill_between(np.arange(len(yte)),
                        (ypred - 2*ystd)[sidx], (ypred + 2*ystd)[sidx],
                        alpha=0.3, color=col, label='95% CI')
        ax.plot(yte[sidx],  '.', ms=3, color='k', alpha=0.5, label='Actual')
        ax.plot(ypred[sidx], '-', lw=1, color=col, label='GPR mean')
        cov = np.mean(np.abs(yte - ypred) <= 2*ystd)
        ax.set_title(f'{lab}\n95% coverage = {cov:.1%}')
        ax.set_xlabel('Test samples (sorted by σ)')
        ax.set_ylabel(lab)
        ax.legend(fontsize=8)

    plt.tight_layout()
    plt.show()
    print("  Fig 2 displayed.")


# ── Figure 3 : Kernel Analysis & Single-Param ─────────────────────────────────
def fig3_gpr_kernel(res):
    print("[Fig 3] GPR kernel analysis…")
    gpr1, gpr2 = res['gpr1'], res['gpr2']
    Xs, mask   = res['Xs'], res['mask']
    y1_tr, y1_te = res['y1_tr'], res['y1_te']
    yp_avg, ys_avg = res['yp_avg'], res['ys_avg']
    y_avg_te   = res['y_avg_te']
    FEAT_NAMES = res['FEAT_NAMES']
    gpr_x5     = res['gpr_x5']

    r2_avg = r2_score(y_avg_te, yp_avg)
    rmse_avg = np.sqrt(mean_squared_error(y_avg_te, yp_avg))
    r2_x5  = r2_score(y1_te, gpr_x5.predict(Xs[:, 4:5][~mask]))

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle('GPR — Kernel Analysis & Single-Parameter Exploration',
                 fontsize=13, fontweight='bold')

    # Panel 0 : ARD relevance scores  1/ℓ²
    k1_ls  = gpr1.kernel_.k1.k2.length_scale
    k2_ls  = gpr2.kernel_.k1.k2.length_scale
    bar_x  = np.arange(8)
    w = 0.35
    axes[0].bar(bar_x - w/2, 1/k1_ls**2, width=w,
                color=BLUE, alpha=0.8, label='Y1 Heating')
    axes[0].bar(bar_x + w/2, 1/k2_ls**2, width=w,
                color=RED,  alpha=0.8, label='Y2 Cooling')
    axes[0].set_xticks(bar_x)
    axes[0].set_xticklabels(FEAT_NAMES, fontsize=7, rotation=30, ha='right')
    axes[0].set_title('ARD Feature Relevance (1/ℓ²)\nHigher = More Relevant')
    axes[0].set_ylabel('Relevance score')
    axes[0].legend()

    # Panel 1 : 1-D GPR  X5 → Y1
    x5_range = np.linspace(Xs[:, 4].min(), Xs[:, 4].max(), 300).reshape(-1, 1)
    mu, sig  = gpr_x5.predict(x5_range, return_std=True)
    axes[1].scatter(Xs[:, 4][~mask], y1_te, s=15, color='k',
                    alpha=0.4, label='Test data')
    axes[1].plot(x5_range, mu, color=BLUE, lw=2, label='GPR mean')
    axes[1].fill_between(x5_range.ravel(), mu - 2*sig, mu + 2*sig,
                         alpha=0.25, color=BLUE, label='95% CI')
    axes[1].set_xlabel('X5  Overall Height (standardised)')
    axes[1].set_ylabel('Heating Load Y1')
    axes[1].set_title(f'Single-Feature GPR: X5 → Y1\nR²={r2_x5:.3f}')
    axes[1].legend(fontsize=8)

    # Panel 2 : Combined target  Y_avg predicted vs actual
    mn, mx = y_avg_te.min(), y_avg_te.max()
    sc = axes[2].scatter(y_avg_te, yp_avg, c=ys_avg,
                         cmap='viridis', s=20, alpha=0.7)
    axes[2].plot([mn, mx], [mn, mx], 'k--', lw=1.5, label='Perfect fit')
    plt.colorbar(sc, ax=axes[2], label='GPR σ')
    axes[2].set_xlabel('Actual  (Y1+Y2)/2')
    axes[2].set_ylabel('Predicted')
    axes[2].set_title(f'Single-Output GPR on Y=(Y1+Y2)/2\n'
                      f'R²={r2_avg:.4f}   RMSE={rmse_avg:.3f}')
    axes[2].legend(fontsize=8)

    plt.tight_layout()
    plt.show()
    print("  Fig 3 displayed.")


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 2  ─  LINEAR REGRESSION  (Green Building)
# ══════════════════════════════════════════════════════════════════════════════

def run_linear_regression(gb):
    """Feature selection and linear regression on the green building dataset."""
    y_gb      = gb['predicted_energy_demand'].values
    feat_cols = [c for c in gb.columns if c != 'predicted_energy_demand']
    X_gb      = gb[feat_cols].values

    # Correlation-based feature selection  |r| > 0.05
    corr_gb          = gb.corr()
    corr_with_target = (corr_gb['predicted_energy_demand']
                        .drop('predicted_energy_demand')
                        .abs()
                        .sort_values(ascending=False))
    selected = corr_with_target[corr_with_target > 0.05].index.tolist()
    print(f"\n  Selected features ({len(selected)}): {selected}")
    print("  Correlation with target:\n",
          corr_with_target.round(4).to_string())

    X_sel = gb[selected].values
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_sel, y_gb, test_size=0.2, random_state=42)

    scaler_lr  = StandardScaler()
    X_tr_s     = scaler_lr.fit_transform(X_tr)
    X_te_s     = scaler_lr.transform(X_te)

    lr = LinearRegression()
    lr.fit(X_tr_s, y_tr)
    y_pred = lr.predict(X_te_s)

    r2   = r2_score(y_te, y_pred)
    rmse = np.sqrt(mean_squared_error(y_te, y_pred))
    mae  = mean_absolute_error(y_te, y_pred)

    # Full model (all features) for comparison
    sc_full = StandardScaler()
    X_tr_f, X_te_f, y_tr_f, y_te_f = train_test_split(
        X_gb, y_gb, test_size=0.2, random_state=42)
    lr_full = LinearRegression()
    lr_full.fit(sc_full.fit_transform(X_tr_f), y_tr_f)
    y_pred_full = lr_full.predict(sc_full.transform(X_te_f))
    r2_full   = r2_score(y_te_f, y_pred_full)
    rmse_full = np.sqrt(mean_squared_error(y_te_f, y_pred_full))

    # Cross-validation
    cv_sel = cross_val_score(LinearRegression(),
                             scaler_lr.fit_transform(X_sel), y_gb,
                             cv=5, scoring='r2')
    cv_all = cross_val_score(LinearRegression(),
                             sc_full.fit_transform(X_gb), y_gb,
                             cv=5, scoring='r2')

    coef_df = pd.DataFrame({'Feature': selected, 'Coefficient': lr.coef_})
    coef_df = coef_df.reindex(coef_df['Coefficient'].abs().sort_values(ascending=False).index)

    print(f"\n  LR selected ({len(selected)} feat): "
          f"R²={r2:.4f}  RMSE={rmse:.3f}  MAE={mae:.3f}")
    print(f"  LR all ({len(feat_cols)} feat):       "
          f"R²={r2_full:.4f}  RMSE={rmse_full:.3f}")
    print(f"  5-Fold CV (selected): "
          f"mean={cv_sel.mean():.4f} ±{cv_sel.std():.4f}")

    return dict(
        gb=gb, y_gb=y_gb, feat_cols=feat_cols, X_gb=X_gb,
        selected=selected, corr_with_target=corr_with_target,
        y_te=y_te, y_pred=y_pred,
        r2=r2, rmse=rmse, mae=mae,
        r2_full=r2_full, rmse_full=rmse_full,
        coef_df=coef_df, cv_sel=cv_sel, cv_all=cv_all,
        scaler_lr=scaler_lr, X_sel=X_sel,
        sc_full=sc_full, X_gb_full=X_gb,
    )


# ── Figure 4 : Green Building EDA ─────────────────────────────────────────────
def fig4_gb_eda(lr_res):
    print("[Fig 4] Green Building EDA…")
    gb, y_gb, feat_cols = lr_res['gb'], lr_res['y_gb'], lr_res['feat_cols']

    fig, axes = plt.subplots(3, 4, figsize=(16, 11))
    fig.suptitle('Green Building Dataset — EDA', fontsize=14, fontweight='bold')

    for i, col in enumerate(feat_cols):
        r, c = divmod(i, 4)
        ax   = axes[r][c]
        ax.scatter(gb[col], y_gb, alpha=0.15, s=8, color=GREEN)
        cc = np.corrcoef(gb[col], y_gb)[0, 1]
        ax.set_xlabel(col, fontsize=8)
        ax.set_ylabel('Energy Demand' if c == 0 else '', fontsize=7)
        ax.set_title(f'r = {cc:.3f}', fontsize=9)

    plt.tight_layout()
    plt.show()
    print("  Fig 4 displayed.")


# ── Figure 5 : Linear Regression Results ─────────────────────────────────────
def fig5_lr_results(lr_res):
    print("[Fig 5] Linear Regression results…")
    y_te             = lr_res['y_te']
    y_pred           = lr_res['y_pred']
    coef_df          = lr_res['coef_df']
    corr_with_target = lr_res['corr_with_target']
    r2, rmse, mae    = lr_res['r2'], lr_res['rmse'], lr_res['mae']

    res_lr = y_pred - y_te

    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    fig.suptitle('Linear Regression — Green Building Energy Demand',
                 fontsize=14, fontweight='bold')

    # [0,0] Predicted vs Actual
    mn, mx = y_te.min(), y_te.max()
    axes[0][0].scatter(y_te, y_pred, alpha=0.3, s=15, color=GREEN)
    axes[0][0].plot([mn, mx], [mn, mx], 'k--', lw=1.5, label='Perfect fit')
    axes[0][0].set_xlabel('Actual')
    axes[0][0].set_ylabel('Predicted')
    axes[0][0].set_title(f'Predicted vs Actual\nR²={r2:.4f}   RMSE={rmse:.3f}')
    axes[0][0].legend()

    # [0,1] Residuals vs Fitted
    axes[0][1].scatter(y_pred, res_lr, alpha=0.3, s=12, color=GREEN)
    axes[0][1].axhline(0, color='k', ls='--', lw=1.5)
    axes[0][1].set_xlabel('Fitted Values')
    axes[0][1].set_ylabel('Residuals')
    axes[0][1].set_title('Residuals vs Fitted')

    # [0,2] Residual distribution + Normal overlay
    axes[0][2].hist(res_lr, bins=40, color=GREEN,
                    alpha=0.7, edgecolor='white', density=True)
    xx = np.linspace(res_lr.min(), res_lr.max(), 200)
    axes[0][2].plot(xx, sp_norm.pdf(xx, res_lr.mean(), res_lr.std()),
                    'k-', lw=2, label='Normal fit')
    axes[0][2].set_title('Residual Distribution')
    axes[0][2].set_xlabel('Residual')
    axes[0][2].legend()

    # [1,0] Standardised coefficients
    colors = [GREEN if v > 0 else RED for v in coef_df['Coefficient']]
    axes[1][0].barh(coef_df['Feature'], coef_df['Coefficient'],
                    color=colors, alpha=0.8)
    axes[1][0].axvline(0, color='k', lw=1)
    axes[1][0].set_title('Standardised Coefficients')
    axes[1][0].set_xlabel('Coefficient')
    axes[1][0].tick_params(labelsize=8)

    # [1,1] |r| bar chart with selection threshold
    axes[1][1].bar(corr_with_target.index, corr_with_target.values,
                   color=[GREEN if v > 0.1 else GRAY
                          for v in corr_with_target.values],
                   alpha=0.8)
    axes[1][1].axhline(0.05, color=RED, ls='--', lw=1,
                       label='Selection threshold (0.05)')
    axes[1][1].set_title('|Pearson r| with Energy Demand')
    axes[1][1].set_xlabel('Feature')
    axes[1][1].set_ylabel('|r|')
    axes[1][1].tick_params(rotation=45, labelsize=7)
    axes[1][1].legend(fontsize=8)

    # [1,2] Actual vs Predicted — 100 sample overlay
    rng    = np.random.RandomState(7)
    isamp  = np.sort(rng.choice(len(y_te), 100, replace=False))
    axes[1][2].plot(y_te[isamp],   'o', ms=4, color='k',
                    alpha=0.6, label='Actual')
    axes[1][2].plot(y_pred[isamp], 's', ms=4, color=GREEN,
                    alpha=0.7, label='Predicted')
    axes[1][2].set_xlabel('Sample index')
    axes[1][2].set_ylabel('Energy Demand')
    axes[1][2].set_title('Actual vs Predicted (100 samples)')
    axes[1][2].legend(fontsize=8)

    plt.tight_layout()
    plt.show()
    print("  Fig 5 displayed.")


# ── Figure 6 : LR Diagnostics — heatmap + CV box-plot ────────────────────────
def fig6_lr_diagnostics(lr_res):
    print("[Fig 6] LR diagnostics…")
    gb               = lr_res['gb']
    selected         = lr_res['selected']
    cv_sel, cv_all   = lr_res['cv_sel'], lr_res['cv_all']

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle('Linear Regression — Model Diagnostics',
                 fontsize=13, fontweight='bold')

    # Panel 0 : correlation heatmap among selected features + target
    sns.heatmap(
        gb[selected + ['predicted_energy_demand']].corr(),
        ax=axes[0], cmap='RdBu_r', vmin=-1, vmax=1,
        annot=True, fmt='.2f', annot_kws={'size': 7},
        linewidths=0.3, cbar_kws={'shrink': 0.7})
    axes[0].set_title('Correlation: Selected Features + Target')
    axes[0].tick_params(labelsize=7, rotation=45)

    # Panel 1 : 5-Fold CV R² box-plot
    bp = axes[1].boxplot(
        [cv_sel, cv_all],
        labels=['Selected\n(4 feat)', 'All\n(12 feat)'],
        patch_artist=True,
        medianprops={'color': 'k', 'lw': 2})
    for patch, color in zip(bp['boxes'], [GREEN, BLUE]):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    axes[1].set_title('5-Fold CV R² Distribution')
    axes[1].set_ylabel('R²')
    axes[1].set_ylim(0, 1.05)
    for i, (scores, c) in enumerate([(cv_sel, GREEN), (cv_all, BLUE)], 1):
        axes[1].text(i, scores.mean() + 0.02,
                     f'μ={scores.mean():.3f}',
                     ha='center', fontsize=9, color=c, fontweight='bold')

    plt.tight_layout()
    plt.show()
    print("  Fig 6 displayed.")


# ══════════════════════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════════════════════
if __name__ == '__main__':
    print("=" * 65)
    print("  PART 1 — GAUSSIAN PROCESS REGRESSION  (ENB2012)")
    print("=" * 65)
    df_enb = load_enb2012()
    print(f"  Dataset loaded: {df_enb.shape[0]} rows × {df_enb.shape[1]} cols")

    gpr_res = run_gpr(df_enb)

    fig1_eda(df_enb, gpr_res)
    fig2_gpr_results(gpr_res)
    fig3_gpr_kernel(gpr_res)

    print("\n" + "=" * 65)
    print("  PART 2 — LINEAR REGRESSION  (Green Building)")
    print("=" * 65)
    df_gb = load_green_building()
    print(f"  Dataset loaded: {df_gb.shape[0]} rows × {df_gb.shape[1]} cols")

    lr_res = run_linear_regression(df_gb)

    fig4_gb_eda(lr_res)
    fig5_lr_results(lr_res)
    fig6_lr_diagnostics(lr_res)

    print("\n" + "=" * 65)
    print("  ALL DONE — all 6 figures displayed.")
    print("=" * 65)

  PART 1 — GAUSSIAN PROCESS REGRESSION  (ENB2012)
  Dataset loaded: 768 rows × 10 cols
  Fitting GPR for Y1 (Heating Load)…
  Fitting GPR for Y2 (Cooling Load)…
  Fitting GPR for Y_avg = (Y1+Y2)/2…

  GPR Y1 : R²=0.9987  RMSE=0.494  MAE=0.388
  GPR Y2 : R²=0.9961  RMSE=0.506  MAE=0.406
  GPR avg: R²=0.9989  RMSE=0.352

[Fig 1] ENB2012 EDA…
  Fig 1 displayed.
[Fig 2] GPR results…
  Fig 2 displayed.
[Fig 3] GPR kernel analysis…
  Fig 3 displayed.

  PART 2 — LINEAR REGRESSION  (Green Building)
  Dataset loaded: 2400 rows × 13 cols

  Selected features (5): ['hvac_energy', 'floor_area', 'temperature', 'insulation_thickness', 'solar_radiation']
  Correlation with target:
 hvac_energy             0.9575
floor_area              0.1571
temperature             0.1345
insulation_thickness    0.1299
solar_radiation         0.0534
occupancy_count         0.0475
window_to_wall_ratio    0.0369
building_age            0.0350
light_level             0.0178
co2_level               0.0151
humidity     



---

## Part 1 — Gaussian Process Regression (ENB2012)

### Dataset & EDA (Fig 1)

The ENB2012 dataset contains 768 samples of 12 building shapes, each described by eight features (X1–X8) and two targets: Heating Load (Y1) and Cooling Load (Y2), both in kWh/m².

Key observations from EDA:
- **X1 (Relative Compactness)** and **X5 (Overall Height)** are the strongest linear predictors of both Y1 and Y2 ($|r| > 0.85$).
- **X3 (Wall Area)** and **X2 (Surface Area)** also show strong correlation.
- X6 (Orientation) is essentially uncorrelated with either target — consistent with the literature.
- Y1 and Y2 are themselves moderately correlated ($r \approx 0.97$ in the synthetic data), which motivates the single-parameter exploration below.

### GPR Setup

A **Matérn-5/2 kernel** with automatic relevance determination (ARD — one length-scale per feature) was chosen, plus a WhiteKernel noise term:

```latex
k(x,x') = \sigma^2 \cdot \text{Matérn}(\nu=2.5, \ell=[\ell_1,...,\ell_8]) + \sigma^2_{\text{noise}} \cdot \delta
```

The ARD length-scales are learned via marginal likelihood maximisation, which effectively performs automatic feature selection. The data was standardised before fitting. Due to GPR's O(n³) cost, 300 training points were used, with 468 held out for evaluation.

### Results (Fig 2)

| Target | R² | RMSE | MAE |
|---|---|---|---|
| Y1 Heating Load | **0.9987** | 0.493 kWh/m² | 0.388 |
| Y2 Cooling Load | **0.9961** | 0.503 kWh/m² | 0.405 |
| Y_avg = (Y1+Y2)/2 | **0.9989** | 0.351 | — |

The GPR fits both loads near-perfectly. Residuals are unstructured and small. The 95% uncertainty bands achieved empirical coverage very close to the nominal 95%, confirming the GPR is well-calibrated.

### Kernel Analysis (Fig 3)

Inspecting the learned ARD length-scales (plotted as relevance = $1/\ell^2$):
- **X5 (Overall Height)** and **X1 (Compactness)** emerge as the most relevant features — consistent with physical intuition (taller buildings with compact shapes lose/gain heat differently).
- X6 (Orientation) and X4 (Roof Area) received very large length-scales, meaning the kernel effectively ignores them — a powerful automatic feature selection.

### Single-Parameter GPR Discussion

"Single-parameter" can mean two things; both were explored:

**Single-feature GPR (one-dimensional input):**
Using only X5 (Overall Height) gives $R^2 \approx 0.955$ on Y1 — remarkably good for a single feature. X1 (Compactness) alone gives $R^2 \approx 0.996$. X7 (Glazing Area) alone performs poorly ($R^2 \approx -0.006$), confirming glazing area alone has very low predictive power without context. This demonstrates GPR's ability to detect non-linear input relevance through kernel learning.

**Single-output GPR on a combined target Y = (Y1+Y2)/2:**
Since Y1 and Y2 are highly correlated ($r \approx 0.97$), we can collapse them into one scalar. The GPR on Y_avg achieves $R^2 = 0.9989$ and RMSE = 0.35 — slightly better than the individual models, because averaging reduces noise variance in the target.

**Conclusion on GPR:** GPR is an excellent model for this dataset. Its non-parametric Bayesian nature means it naturally adapts to the non-linear relationships present (particularly the interplay between compactness and height). The ARD kernel provides built-in feature importance ranking. The probabilistic output (posterior mean + variance) gives honest uncertainty estimates — valuable in building energy simulation where safety margins matter. The only limitation is scalability: for the full 768-sample dataset without subsampling, training takes considerable time.

---

## Part 2 — Linear Regression (Green Building Dataset)

### Dataset

The green building dataset has 2,400 samples and 12 features covering temperature, humidity, CO₂, light, occupancy, HVAC energy, solar radiation, wind speed, building age, insulation thickness, window-to-wall ratio, and floor area. The target is `predicted_energy_demand`.

### Feature Selection (Fig 4 & 6)

Pearson correlations with the target were computed for all 12 features:

| Feature | $|r|$ | Selected? |
|---|---|---|
| hvac_energy | **0.954** | ✅ |
| insulation_thickness | **0.219** | ✅ |
| temperature | **0.170** | ✅ |
| floor_area | **0.143** | ✅ |
| window_to_wall_ratio | 0.049 | ❌ |
| co2_level, solar_radiation, … | < 0.042 | ❌ |

Features with $|r| < 0.05$ were excluded. Physical justification: HVAC energy directly represents how hard the heating/cooling system is working — the most direct driver of energy demand. Temperature drives heating/cooling need. Insulation and floor area govern heat loss through the envelope. The remaining features (CO₂, occupancy, wind speed, etc.) do not have a direct thermodynamic link to total energy demand and their near-zero correlations confirm they add no linear signal.

The selected features also have acceptably low inter-correlation (see heatmap, Fig 6), avoiding multicollinearity issues.

### Results (Fig 5 & 6)

| Model | R² | RMSE | MAE |
|---|---|---|---|
| Linear (4 selected features) | **0.9857** | 2.647 kWh | 2.108 |
| Linear (all 12 features) | 0.9899 | 2.227 kWh | — |
| 5-Fold CV (selected) | 0.9837 ± 0.001 | — | — |

The standardised coefficients reveal the hierarchy of influence:

- **HVAC energy** (+20.9): overwhelmingly dominant — a 1$\sigma$ rise in HVAC usage drives energy demand up by ~21 units.
- **Temperature** (+3.5): higher outdoor temperature increases cooling demand.
- **Insulation** (−3.5): better insulation reduces demand (correct physical sign).
- **Floor area** (+3.2): larger buildings consume more energy.

Residuals are approximately normally distributed with mean $\approx 0$ and show no strong heteroscedastic pattern, satisfying key OLS assumptions. The CV results are stable (std = 0.001), indicating no overfitting.

**Conclusion on Linear Regression:** A linear model with just four physically motivated features explains 98.6% of variance in predicted energy demand. The near-identical performance between the 4-feature and 12-feature models ($\Delta R^2 < 0.004$) confirms that the excluded features contribute no meaningful linear signal and should be omitted on grounds of parsimony. The model is well-calibrated, generalisable, and interpretable — making it suitable for practical use in green building design decisions.